# Análise VADER com 6.000 comentários reais

Este notebook foi preparado para gerar uma nova execução completa, com:

- 2.000 comentários de `r/programming`;
- 2.000 comentários de `r/cscareerquestions`;
- 2.000 comentários de `r/devops`;
- período de 2021 a 2024;
- análise VADER com `neg`, `neu`, `pos` e `compound`;
- base anonimizada, tabelas e gráficos.

> A coleta é uma reconstrução técnica posterior. Ela segue a metodologia do TCC, mas não garante que sejam exatamente os mesmos comentários da amostra original.

In [1]:
!pip -q install pandas requests nltk matplotlib python-dateutil

In [2]:
from __future__ import annotations

import hashlib
import json
import random
import time
import zipfile
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import nltk
import pandas as pd
import requests
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download("vader_lexicon", quiet=True)

BASE = Path("/content/vader_6000")
PUBLICO = BASE / "arquivos_para_github"
PRIVADO = BASE / "base_completa_privada"
PUBLICO.mkdir(parents=True, exist_ok=True)
PRIVADO.mkdir(parents=True, exist_ok=True)

API_URL = "https://arctic-shift.photon-reddit.com/api/comments/search"
SUBREDDITS = ["programming", "cscareerquestions", "devops"]
INICIO = datetime(2021, 1, 1, tzinfo=timezone.utc)
FIM = datetime(2025, 1, 1, tzinfo=timezone.utc)  # exclusivo
META = 2000
SEMENTE = 2026
MIN_CARACTERES = 40
PAUSA = 0.35
TENTATIVAS = 5

TERMOS = [
    "burnout", "burning out", "stress", "stressed", "stressful",
    "deadline", "deadlines", "overworked", "workload", "overtime",
    "tired", "exhausted", "exhausting", "exhaustion", "drained",
    "anxious", "anxiety", "pressure", "on call", "on-call",
    "incident", "production issue", "production issues", "outage",
    "bug", "crash", "overload", "can't keep up", "cannot keep up"
]

print("Configuração carregada.")

Configuração carregada.


## 1. Coleta adaptativa

A coleta começa com janelas semanais. Se uma semana retornar muitos comentários, o código divide essa semana em dias para aumentar a cobertura. A amostra final é aleatória e equilibrada por comunidade.

In [3]:
def extrair_lista(payload: Any) -> list[dict[str, Any]]:
    if isinstance(payload, list):
        return [x for x in payload if isinstance(x, dict)]
    if isinstance(payload, dict):
        for chave in ("data", "results", "comments"):
            valor = payload.get(chave)
            if isinstance(valor, list):
                return [x for x in valor if isinstance(x, dict)]
    return []


def requisitar(
    session: requests.Session,
    subreddit: str,
    inicio: datetime,
    fim: datetime,
    ordem: str,
) -> list[dict[str, Any]]:
    params = {
        "subreddit": subreddit,
        "after": inicio.date().isoformat(),
        "before": fim.date().isoformat(),
        "limit": 100,
        "sort": ordem,
        "fields": "id,subreddit,created_utc,body",
    }

    ultimo_erro = None
    for tentativa in range(1, TENTATIVAS + 1):
        try:
            resposta = session.get(API_URL, params=params, timeout=90)
            if resposta.status_code == 429:
                time.sleep(min(60, tentativa * 5))
                continue
            resposta.raise_for_status()
            return extrair_lista(resposta.json())
        except Exception as erro:
            ultimo_erro = erro
            time.sleep(min(30, 2 ** tentativa))

    print(f"Falha na janela {inicio.date()}–{fim.date()}: {ultimo_erro}")
    return []


def preparar(item: dict[str, Any]) -> dict[str, Any] | None:
    corpo = str(item.get("body") or "").strip()
    if not corpo or corpo in {"[deleted]", "[removed]"}:
        return None
    if len(corpo) < MIN_CARACTERES:
        return None

    try:
        created = int(float(item.get("created_utc")))
        data = datetime.fromtimestamp(created, tz=timezone.utc).isoformat()
    except Exception:
        return None

    identificador = str(item.get("id") or f"{data}|{corpo}")
    return {
        "comment_id_hash": hashlib.sha256(identificador.encode("utf-8")).hexdigest()[:16],
        "subreddit": str(item.get("subreddit") or "").lower(),
        "created_utc": data,
        "body_original": corpo,
        "fonte_reconstrucao": "Arctic Shift API",
    }


def coletar_subreddit(subreddit: str) -> pd.DataFrame:
    session = requests.Session()
    session.headers.update({
        "User-Agent": "PesquisaAcademicaVADER/1.0 (uso educacional)"
    })

    registros: dict[str, dict[str, Any]] = {}
    atual = INICIO
    semana = 0

    while atual < FIM:
        proxima = min(atual + timedelta(days=7), FIM)
        ordem = "asc" if semana % 2 == 0 else "desc"
        itens = requisitar(session, subreddit, atual, proxima, ordem)

        # Se a semana encostar no limite, divide em dias para melhorar a cobertura.
        if len(itens) >= 95:
            itens = []
            dia = atual
            indice_dia = 0
            while dia < proxima:
                fim_dia = min(dia + timedelta(days=1), proxima)
                ordem_dia = "asc" if indice_dia % 2 == 0 else "desc"
                itens.extend(requisitar(session, subreddit, dia, fim_dia, ordem_dia))
                dia = fim_dia
                indice_dia += 1
                time.sleep(PAUSA)

        for item in itens:
            registro = preparar(item)
            if registro:
                registros[registro["comment_id_hash"]] = registro

        semana += 1
        if semana % 25 == 0:
            print(f"r/{subreddit}: {semana} semanas; {len(registros)} comentários válidos.")

        atual = proxima
        time.sleep(PAUSA)

    df = pd.DataFrame(registros.values())
    if len(df) < META:
        raise RuntimeError(
            f"r/{subreddit} retornou apenas {len(df)} comentários válidos. "
            "Execute novamente mais tarde; a API pode estar limitando respostas."
        )

    return df


todos = []
for subreddit in SUBREDDITS:
    print(f"\nColetando r/{subreddit}...")
    df_sub = coletar_subreddit(subreddit)
    print(f"r/{subreddit}: {len(df_sub)} candidatos válidos.")
    todos.append(df_sub)

candidatos = pd.concat(todos, ignore_index=True)
print("\nColeta concluída.")


Coletando r/programming...
r/programming: 25 semanas; 14216 comentários válidos.
r/programming: 50 semanas; 28430 comentários válidos.
r/programming: 75 semanas; 42544 comentários válidos.
r/programming: 100 semanas; 56792 comentários válidos.
r/programming: 125 semanas; 71029 comentários válidos.
r/programming: 150 semanas; 83924 comentários válidos.
r/programming: 175 semanas; 98959 comentários válidos.
r/programming: 200 semanas; 113852 comentários válidos.
r/programming: 119009 candidatos válidos.

Coletando r/cscareerquestions...
r/cscareerquestions: 25 semanas; 13965 comentários válidos.
r/cscareerquestions: 50 semanas; 27664 comentários válidos.
r/cscareerquestions: 75 semanas; 41580 comentários válidos.
r/cscareerquestions: 100 semanas; 55679 comentários válidos.
r/cscareerquestions: 125 semanas; 69749 comentários válidos.
r/cscareerquestions: 150 semanas; 84096 comentários válidos.
r/cscareerquestions: 175 semanas; 98756 comentários válidos.
r/cscareerquestions: 200 semanas; 

## 2. Seleção exata de 6.000 comentários e aplicação do VADER

In [4]:
partes = []
for indice, subreddit in enumerate(SUBREDDITS):
    grupo = candidatos[candidatos["subreddit"] == subreddit].drop_duplicates("comment_id_hash")
    selecionado = grupo.sample(n=META, random_state=SEMENTE + indice)
    partes.append(selecionado)

amostra = pd.concat(partes, ignore_index=True)
amostra = amostra.sample(frac=1, random_state=SEMENTE).reset_index(drop=True)

assert len(amostra) == 6000, f"Total inesperado: {len(amostra)}"
assert amostra.groupby("subreddit").size().to_dict() == {
    "programming": 2000,
    "cscareerquestions": 2000,
    "devops": 2000,
}

analisador = SentimentIntensityAnalyzer()
pontos = amostra["body_original"].apply(analisador.polarity_scores).apply(pd.Series)

for coluna in ["neg", "neu", "pos", "compound"]:
    amostra[coluna] = pontos[coluna].astype(float)

def classificar(valor: float) -> str:
    if valor >= 0.05:
        return "positivo"
    if valor <= -0.05:
        return "negativo"
    return "neutro"

def termos_encontrados(texto: str) -> str:
    texto = texto.lower()
    return "; ".join([termo for termo in TERMOS if termo in texto])

amostra["classe_sentimento"] = amostra["compound"].apply(classificar)
amostra["termos_estresse"] = amostra["body_original"].apply(termos_encontrados)

print(amostra.groupby("subreddit").size())
print("Total final:", len(amostra))

subreddit
cscareerquestions    2000
devops               2000
programming          2000
dtype: int64
Total final: 6000


## 3. Geração dos arquivos completos

In [5]:
# Base integral privada, com textos.
amostra.to_csv(
    PRIVADO / "amostra_reddit_6000_completa.csv",
    index=False,
    encoding="utf-8-sig",
)

# Resultados públicos sem o texto integral.
publico = amostra[
    [
        "comment_id_hash", "subreddit", "created_utc",
        "neg", "neu", "pos", "compound",
        "classe_sentimento", "termos_estresse",
        "fonte_reconstrucao",
    ]
].copy()

publico.to_csv(
    PUBLICO / "resultados_vader_6000.csv",
    index=False,
    encoding="utf-8-sig",
)

resumo = (
    amostra.groupby(["subreddit", "classe_sentimento"])
    .size()
    .rename("quantidade")
    .reset_index()
)
resumo["percentual"] = (
    resumo["quantidade"]
    / resumo.groupby("subreddit")["quantidade"].transform("sum")
    * 100
).round(2)

total = (
    amostra.groupby("classe_sentimento")
    .size()
    .rename("quantidade")
    .reset_index()
)
total["subreddit"] = "TOTAL"
total["percentual"] = (total["quantidade"] / len(amostra) * 100).round(2)

resumo_final = pd.concat([resumo, total], ignore_index=True)
resumo_final.to_csv(
    PUBLICO / "resumo_sentimentos_6000.csv",
    index=False,
    encoding="utf-8-sig",
)

from collections import Counter
contador = Counter()
for celula in amostra["termos_estresse"].fillna("").astype(str):
    for termo in [x.strip() for x in celula.split(";") if x.strip()]:
        contador[termo] += 1

frequencia = pd.DataFrame(
    contador.most_common(),
    columns=["termo", "quantidade_de_comentarios"],
)
if not frequencia.empty:
    frequencia["percentual_da_amostra"] = (
        frequencia["quantidade_de_comentarios"] / len(amostra) * 100
    ).round(2)

frequencia.to_csv(
    PUBLICO / "frequencia_termos_6000.csv",
    index=False,
    encoding="utf-8-sig",
)

# Gráfico geral.
ordem = ["negativo", "neutro", "positivo"]
dist = (
    total.set_index("classe_sentimento")["percentual"]
    .reindex(ordem)
    .fillna(0)
)
plt.figure(figsize=(8, 5))
dist.plot(kind="bar")
plt.xlabel("Classificação")
plt.ylabel("Percentual")
plt.title("Distribuição dos sentimentos segundo o VADER")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(PUBLICO / "grafico_sentimentos_6000.png", dpi=200)
plt.close()

# Gráfico de termos.
if not frequencia.empty:
    top = frequencia.head(12).sort_values("quantidade_de_comentarios")
    plt.figure(figsize=(9, 6))
    plt.barh(top["termo"], top["quantidade_de_comentarios"])
    plt.xlabel("Quantidade de comentários")
    plt.ylabel("Termo")
    plt.title("Termos associados ao estresse mais recorrentes")
    plt.tight_layout()
    plt.savefig(PUBLICO / "grafico_termos_6000.png", dpi=200)
    plt.close()

print("Arquivos gerados.")
display(resumo_final)
display(frequencia.head(15))

Arquivos gerados.


,subreddit,classe_sentimento,quantidade,percentual
0,cscareerquestions,negativo,459,22.95
1,cscareerquestions,neutro,283,14.15
2,cscareerquestions,positivo,1258,62.90
3,devops,negativo,402,20.10
4,devops,neutro,336,16.80
5,devops,positivo,1262,63.10
6,programming,negativo,566,28.30
7,programming,neutro,343,17.15
8,programming,positivo,1091,54.55
9,TOTAL,negativo,1427,23.78


,termo,quantidade_de_comentarios,percentual_da_amostra
0,bug,94,1.57
1,stress,38,0.63
2,incident,24,0.40
3,tired,22,0.37
4,crash,16,0.27
5,workload,15,0.25
6,outage,15,0.25
7,anxiety,14,0.23
8,deadline,14,0.23
9,pressure,13,0.22


## 4. Documentação e download

In [6]:
readme = """
# Análise VADER com 6.000 comentários

Este repositório contém uma execução completa com 6.000 comentários reais,
distribuídos igualmente entre:

- r/programming: 2.000;
- r/cscareerquestions: 2.000;
- r/devops: 2.000.

Período: 2021 a 2024.

Técnica: VADER, com indicadores `neg`, `neu`, `pos` e `compound`.

## Arquivos

- `resultados_vader_6000.csv`;
- `resumo_sentimentos_6000.csv`;
- `frequencia_termos_6000.csv`;
- `grafico_sentimentos_6000.png`;
- `grafico_termos_6000.png`;
- `Executar_VADER_6000_Comentarios_Colab.ipynb`.

## Observação

A coleta foi reconstruída posteriormente com acesso histórico via Arctic Shift.
Ela segue o recorte metodológico do TCC, mas não garante os mesmos comentários
da amostra original não preservada.
""".strip()

(PUBLICO / "README.md").write_text(readme, encoding="utf-8")

registro = {
    "data_execucao_utc": datetime.now(timezone.utc).isoformat(),
    "total": len(amostra),
    "por_subreddit": amostra.groupby("subreddit").size().to_dict(),
    "periodo": "2021-01-01 a 2024-12-31",
    "metodo": "VADER",
    "fonte_reconstrucao": "Arctic Shift API",
}
(PUBLICO / "registro_execucao_6000.json").write_text(
    json.dumps(registro, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

zip_publico = Path("/content/GITHUB_COMPLETO_VADER_6000.zip")
zip_privado = Path("/content/BASE_PRIVADA_VADER_6000_NAO_PUBLICAR.zip")

with zipfile.ZipFile(zip_publico, "w", zipfile.ZIP_DEFLATED) as zf:
    for arquivo in sorted(PUBLICO.iterdir()):
        if arquivo.is_file():
            zf.write(arquivo, arcname=arquivo.name)

with zipfile.ZipFile(zip_privado, "w", zipfile.ZIP_DEFLATED) as zf:
    for arquivo in sorted(PRIVADO.iterdir()):
        if arquivo.is_file():
            zf.write(arquivo, arcname=arquivo.name)

print("Pacotes criados.")

Pacotes criados.


In [7]:
from google.colab import files
files.download("/content/GITHUB_COMPLETO_VADER_6000.zip")
files.download("/content/BASE_PRIVADA_VADER_6000_NAO_PUBLICAR.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>